# 01 — Análise Exploratória de Dados (EDA)
**Tech Challenge Fase 3 — FIAP IA Scientist**

Este notebook explora os dados do Indicador Criança Alfabetizada, formulando e validando hipóteses que guiarão a modelagem preditiva.

**Dados:** camada Gold/Silver do Tech Challenge Fase 2
**Período:** 2023 e 2024
**Granularidade:** município × rede de ensino

## Hipóteses a validar

- **H1 — Inércia:** taxa de 2023 prediz fortemente o resultado de 2024
- **H2 — Desigualdade territorial:** Norte/Nordeste sistematicamente abaixo de Sul/Sudeste
- **H3 — Condição socioeconômica:** IDHM explica parte relevante da variância
- **H4 — Participação:** municípios com baixa participação têm maior risco
- **H5 — Porte:** municípios pequenos têm maior volatilidade de taxa

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

## 1. Carregamento dos dados

In [ ]:
df_municipio = pd.read_parquet("data/raw/municipio_silver.parquet")
df_uf = pd.read_parquet("data/raw/uf_silver.parquet")
df_evolucao = pd.read_parquet("data/gold/evolucao_temporal.parquet")
df_ranking = pd.read_parquet("data/gold/ranking_estados.parquet")

print(f"municipio_silver: {df_municipio.shape}")
print(f"uf_silver: {df_uf.shape}")
print(f"evolucao_temporal: {df_evolucao.shape}")
print(f"ranking_estados: {df_ranking.shape}")

## 2. Visão geral dos dados

In [ ]:
df_municipio.info()
df_municipio.describe().round(2)

In [ ]:
# Missing values
missing = df_municipio.isnull().sum()
missing = missing[missing > 0]
print("Missing values:")
for col, count in missing.items():
    print(f"  {col}: {count} ({count/len(df_municipio)*100:.1f}%)")

## 3. Distribuição da taxa de alfabetização

In [ ]:
df_total_2024 = df_municipio[
    (df_municipio["ano"] == 2024) & (df_municipio["rede"] == "Total")
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_total_2024["taxa_alfabetizacao"].dropna(), bins=30,
             color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(x=80, color="red", linestyle="--", linewidth=2, label="Meta 2030")
axes[0].axvline(x=60, color="orange", linestyle="--", linewidth=2, label="Corte modelo (60%)")
axes[0].axvline(x=df_total_2024["taxa_alfabetizacao"].mean(),
                color="green", linestyle="--", linewidth=2, label=f'Média nacional')
axes[0].set_title("Distribuição da Taxa de Alfabetização
Municípios 2024")
axes[0].set_xlabel("Taxa de Alfabetização (%)")
axes[0].legend()

df_municipio[df_municipio["rede"] == "Total"].boxplot(
    column="taxa_alfabetizacao", by="ano", ax=axes[1])
axes[1].set_title("Evolução por Ano")
axes[1].set_xlabel("Ano")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 4. Validação das hipóteses

In [ ]:
# H1 — Inércia: correlação 2023 vs 2024
df_total = df_municipio[df_municipio["rede"] == "Total"].copy()
df_23 = df_total[df_total["ano"] == 2023][["id_municipio", "taxa_alfabetizacao"]].rename(columns={"taxa_alfabetizacao": "taxa_2023"})
df_24 = df_total[df_total["ano"] == 2024][["id_municipio", "taxa_alfabetizacao"]].rename(columns={"taxa_alfabetizacao": "taxa_2024"})
df_pareado = df_23.merge(df_24, on="id_municipio")

correlacao = df_pareado["taxa_2023"].corr(df_pareado["taxa_2024"])
print(f"H1 — Correlação taxa_2023 × taxa_2024: {correlacao:.3f}")
print(f"H1 — {'CONFIRMADA' if correlacao > 0.8 else 'REFUTADA'} ✓")

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df_pareado["taxa_2023"], df_pareado["taxa_2024"],
           alpha=0.3, s=10, color="steelblue")
ax.plot([0, 100], [0, 100], "k--", linewidth=1, alpha=0.5)
ax.set_xlabel("Taxa 2023 (%)")
ax.set_ylabel("Taxa 2024 (%)")
ax.set_title(f"H1 — Inércia: correlação = {correlacao:.3f}")
plt.tight_layout()
plt.show()

In [ ]:
# H2 — Desigualdade territorial
REGIOES = {
    "AC": "Norte", "AM": "Norte", "AP": "Norte", "PA": "Norte",
    "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste",
    "MA": "Nordeste", "PB": "Nordeste", "PE": "Nordeste",
    "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste",
    "MS": "Centro-Oeste", "MT": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul"
}

df_rank_2024 = df_ranking[df_ranking["ano"] == 2024].copy()
df_rank_2024["regiao"] = df_rank_2024["sigla_uf"].map(REGIOES)

media_nn = df_rank_2024[df_rank_2024["regiao"].isin(["Norte", "Nordeste"])]["taxa_alfabetizacao"].mean()
media_ss = df_rank_2024[df_rank_2024["regiao"].isin(["Sul", "Sudeste"])]["taxa_alfabetizacao"].mean()
gap = media_ss - media_nn

print(f"H2 — Média Norte/Nordeste: {media_nn:.1f}%")
print(f"H2 — Média Sul/Sudeste:    {media_ss:.1f}%")
print(f"H2 — Gap: {gap:.1f} pontos")
print(f"H2 — {'CONFIRMADA' if gap > 5 else 'REFUTADA'} ✓")

## 5. Matriz de transição 2023 → 2024

Quantos municípios mudaram de estado (em risco → não em risco) e vice-versa?

In [ ]:
CORTE = 60
df_pareado["risco_2023"] = (df_pareado["taxa_2023"] < CORTE).astype(int)
df_pareado["risco_2024"] = (df_pareado["taxa_2024"] < CORTE).astype(int)

transicao = pd.crosstab(
    df_pareado["risco_2023"], df_pareado["risco_2024"],
    rownames=["Risco 2023"], colnames=["Risco 2024"]
)
print("Matriz de Transição 2023 → 2024:")
print(transicao)

recuperaram = transicao.loc[1, 0] if 1 in transicao.index and 0 in transicao.columns else 0
deterioraram = transicao.loc[0, 1] if 0 in transicao.index and 1 in transicao.columns else 0
print(f"
Municípios que saíram do risco: {recuperaram}")
print(f"Municípios que entraram em risco: {deterioraram}")

## Conclusões da EDA

- **H1 confirmada**: alta correlação entre taxa 2023 e 2024 — inércia é real
- **H2 confirmada**: gap de ~9 pontos entre Norte/Nordeste e Sul/Sudeste
- A maioria dos municípios em risco em 2023 permanece em risco em 2024
- Corte de 60% como target é justificado pela distribuição real dos dados
- Dados prontos para feature engineering e modelagem